
# OpenAI Responses API & Agents SDK — Practical Code Demo

This notebook follows the practical demo from the lesson.

We will:
1. Make a basic **Responses API** call.
2. Pass structured messages with system and user instructions.
3. Create and run an **OpenAI Agent** using the Agents SDK.
4. Compare the two approaches.
5. Try both synchronous and asynchronous agent execution.

> **Prerequisites:** You need an OpenAI API key and Python 3.9+.


## 1. Install the OpenAI Python Package

In [ ]:
!pip install -U openai


## 2. Configure Your API Key

The OpenAI Python SDK automatically reads the `OPENAI_API_KEY` environment variable.

### Option A — Set it in your operating system

**Windows PowerShell**
```powershell
$env:OPENAI_API_KEY="your-api-key"
```

**macOS/Linux**
```bash
export OPENAI_API_KEY="your-api-key"
```

### Option B — Set it temporarily in Python

Avoid committing API keys to notebooks or source control.


In [ ]:

import os
from openai import OpenAI

# Make sure OPENAI_API_KEY is available in your environment.
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is not set.")

client = OpenAI()

print("OpenAI client created successfully.")


## 3. Simplest Possible Responses API Call

In [ ]:

response = client.responses.create(
    model="gpt-5.5",
    input="What is an AI agent?"
)

print(response.output_text)



### What happened?

The important part is:

```python
client.responses.create(...)
```

We provide:
- `model` — the OpenAI model to use.
- `input` — a simple string containing the user's request.

The generated text can be accessed directly with:

```python
response.output_text
```


## 4. Another Simple Responses API Example

In [ ]:

response = client.responses.create(
    model="gpt-5.5",
    input="Explain the difference between a chatbot and an AI agent."
)

print(response.output_text)


## 5. Responses API with System Instructions and User Messages

In [ ]:

response = client.responses.create(
    model="gpt-5.5",
    input=[
        {
            "role": "system",
            "content": "You are a concise AI tutor. Explain technical concepts using simple examples."
        },
        {
            "role": "user",
            "content": "Differentiate between a chatbot and an AI agent."
        }
    ]
)

print(response.output_text)



### Key Point

The Responses API can accept a simple string for straightforward requests, while structured input can be used when you need to provide roles and instructions.

For this practical demo, the main idea is that the API call remains simple:

```python
response = client.responses.create(...)
```


# Part 2 — OpenAI Agents SDK

## 6. Install the Agents SDK

In [ ]:
!pip install -U openai-agents


The Agents SDK is a higher-level framework for building agents.

It provides an `Agent` abstraction and a `Runner` that manages the agent execution loop.

The basic pattern is:

```text
Define Agent
     ↓
Run Agent with Runner
     ↓
Agent processes the request
     ↓
Return final output
```


## 7. Create Your First Agent

In [ ]:

from agents import Agent, Runner

agent = Agent(
    name="General Knowledge Agent",
    model="gpt-5.5",
    instructions=(
        "You are a helpful general knowledge assistant. "
        "Answer questions clearly and accurately."
    )
)

print("Agent created successfully.")


## 8. Run the Agent Synchronously

In [ ]:

result = Runner.run_sync(
    agent,
    "Who was the first president of the United States?"
)

print(result.final_output)



### What is `Runner.run_sync()` doing?

The Runner handles the execution process for the agent.

Conceptually:

```text
User Question
     ↓
Runner
     ↓
Agent / Model
     ↓
Final Answer
```

The final text response is available through:

```python
result.final_output
```


## 9. Ask Another Question

In [ ]:

result = Runner.run_sync(
    agent,
    "What caused World War I?"
)

print(result.final_output)


## 10. Run Multiple Questions

In [ ]:

questions = [
    "Who was the first president of the United States?",
    "What caused World War I?",
    "What is the difference between machine learning and deep learning?"
]

for question in questions:
    print("=" * 80)
    print("QUESTION:", question)

    result = Runner.run_sync(agent, question)

    print("\nANSWER:")
    print(result.final_output)



## 11. Asynchronous Agent Execution

The Agents SDK also supports asynchronous execution.

The asynchronous equivalent of:

```python
Runner.run_sync(...)
```

is:

```python
await Runner.run(...)
```

This is useful when building applications that use asynchronous Python code, web servers, or multiple concurrent operations.


In [ ]:

import asyncio

async def run_agent_async():
    result = await Runner.run(
        agent,
        "Explain what an AI agent is in three simple sentences."
    )

    print(result.final_output)

await run_agent_async()


# Part 3 — Responses API vs. Agents SDK


## 12. Practical Comparison

### Responses API

You directly call the OpenAI API:

```python
response = client.responses.create(
    model="gpt-5.5",
    input="What is an AI agent?"
)

print(response.output_text)
```

### Agents SDK

You define an agent and let the Runner execute it:

```python
agent = Agent(
    name="General Knowledge Agent",
    model="gpt-5.5",
    instructions="Answer questions clearly."
)

result = Runner.run_sync(
    agent,
    "What is an AI agent?"
)

print(result.final_output)
```

The key difference is the level of abstraction.

**Responses API:** You work more directly with the model API.

**Agents SDK:** You work with an agent abstraction and a Runner that manages the agent execution process.


# Part 4 — Practical Exercise


## Exercise 1 — Responses API

Modify the following example to ask your own question:

```python
response = client.responses.create(
    model="gpt-5.5",
    input="YOUR QUESTION HERE"
)

print(response.output_text)
```

Try:
- Asking for an explanation of MCP.
- Asking for a Python programming example.
- Asking the model to summarize a technical concept.



## Exercise 2 — Agents SDK

Modify the agent instructions so that the agent becomes a Python programming tutor.

Then ask:

> Explain Python decorators with a simple example.

Starter code:

```python
python_tutor = Agent(
    name="Python Tutor",
    model="gpt-5.5",
    instructions="YOUR INSTRUCTIONS HERE"
)

result = Runner.run_sync(
    python_tutor,
    "YOUR QUESTION HERE"
)

print(result.final_output)
```



## Exercise 3 — Compare Both Approaches

Ask the same question using:

1. The Responses API.
2. The Agents SDK.

Compare:
- How much code is required.
- How the model is configured.
- How the input is passed.
- How the output is retrieved.

### Suggested question

> Explain the difference between an AI agent and a traditional chatbot.



# Key Takeaways

- The **Responses API** provides a direct interface for interacting with OpenAI models.
- A basic Responses API request can be made with `client.responses.create()`.
- The generated text can be accessed using `response.output_text`.
- The **Agents SDK** provides a higher-level abstraction for building agents.
- An agent can be created with `Agent(...)`.
- `Runner.run_sync()` executes an agent synchronously.
- `Runner.run()` supports asynchronous execution.
- The practical distinction is that the Responses API gives you a more direct API interface, while the Agents SDK gives you agent-oriented abstractions and execution management.

## Minimal Responses API Example

```python
from openai import OpenAI

client = OpenAI()

response = client.responses.create(
    model="gpt-5.5",
    input="What is an AI agent?"
)

print(response.output_text)
```

## Minimal Agents SDK Example

```python
from agents import Agent, Runner

agent = Agent(
    name="General Knowledge Agent",
    model="gpt-5.5",
    instructions="Answer questions clearly and accurately."
)

result = Runner.run_sync(
    agent,
    "What is an AI agent?"
)

print(result.final_output)
```
